# Stacking: four base models and one logistic combiner

Use the full held-out candidate probabilities from logistic regression, KNN, the decision tree and XGBoost. Reuse the saved trip folds and the shared 32-feature data. Exclude trips `00000033` and `00000054` through the shared loader.

No real fitting starts until all four registry entries are trained and their model, feature-list, full OOF and student-test score files exist. Pilot/subset files do not qualify. This notebook does not build a front end or activate the stacker as the final model.

Meta features, in order: the four per-point base probabilities, raw `distance`, and `candidate_count`. The saved model is a `StandardScaler` + logistic regression pipeline. Its decision scores receive a stable softmax within each GPS point.

In [1]:
%run 06_shared_models.ipynb
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
from time import perf_counter
from datetime import datetime
from zoneinfo import ZoneInfo

base_model_order = ["logistic_regression", "knn", "decision_tree", "xgboost"]
meta_feature_columns = base_model_order + ["distance", "candidate_count"]
registry = json.loads(Path("models/registry.json").read_text())
run_when_ready = False

## Check the incoming artifacts

Every base model must use the same 32-field input order. Full OOF keys and fold assignments are checked against the canonical valid training rows after exclusions. Student-test keys are checked against the complete prediction candidate table, including observations without evaluation labels.

An inner join would hide missing predictions, so the merge below sorts and checks complete key arrays before adding a score column. Invalid folds, duplicates, missing/extra rows, invalid probabilities or non-normalized groups stop fitting.

In [2]:
def missing_base_artifacts(registry):
    missing = []
    for name in base_model_order:
        record = registry["models"].get(name, {})
        if record.get("status") != "trained":
            missing.append(f"{name}: status is {record.get('status', 'missing')}")
        for field in ["model_path", "feature_columns_path", "oof_path", "student_test_scores_path"]:
            path = record.get(field)
            if not path or not Path(path).is_file():
                missing.append(f"{name}: missing {field}")
        path = record.get("feature_columns_path")
        if path and Path(path).is_file():
            if json.loads(Path(path).read_text()) != shared_feature_columns:
                missing.append(f"{name}: feature order differs from the shared 32-field schema")
    return missing


def align_base_scores(source, scores, require_fold=True):
    keys = ["group_id", "segment_id"]
    required = keys + ["score"] + (["fold"] if require_fold else [])
    if not set(required).issubset(scores.columns):
        raise ValueError("Base score file is missing required columns")
    if source.duplicated(keys).any() or scores.duplicated(keys).any():
        raise ValueError("Duplicate candidate keys")
    ordered = scores.sort_values(keys).reset_index(drop=True)
    expected = source.sort_values(keys).reset_index(drop=True)
    if not np.array_equal(ordered[keys].to_numpy(), expected[keys].to_numpy()):
        raise ValueError("Base candidate keys differ: missing, extra or incompatible rows")
    if require_fold and not np.array_equal(ordered["fold"].to_numpy(), expected["fold"].to_numpy()):
        raise ValueError("Base OOF folds differ from the shared fold file")
    probability = pd.to_numeric(ordered["score"], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(probability).all() or not ((probability >= 0) & (probability <= 1)).all():
        raise ValueError("Base scores must be finite probabilities between 0 and 1")
    totals = pd.Series(probability).groupby(ordered["group_id"].to_numpy(), sort=False).sum()
    if not np.allclose(totals, 1, atol=1e-6, rtol=0):
        raise ValueError("Base scores are not normalized within each GPS point")
    return probability


def build_meta_table(source, base_scores, require_fold=True):
    columns = ["trip_id", "group_id", "point_id", "segment_id", "label", "distance", "candidate_count"]
    if require_fold:
        columns.append("fold")
    result = source[columns].sort_values(["group_id", "segment_id"]).reset_index(drop=True).copy()
    for name in base_model_order:
        result[name] = align_base_scores(result, base_scores[name], require_fold)
    assert np.isfinite(result[meta_feature_columns].to_numpy()).all()
    return result


def make_combiner():
    return Pipeline([("scaler", StandardScaler()),
                     ("combiner", LogisticRegression(random_state=seed, max_iter=1000))])

## Train, evaluate and save when all four models are ready

Meta CV uses the same saved folds. The scaler and combiner fit only the other folds' rows. However, these are globally generated base OOF inputs: ordinary second-level CV is **not a fully nested, unbiased assessment** of the whole base-plus-stacker pipeline. Keep that caveat in reports. Final student-test predictions come from base models refitted on all allowed training trips.

Rows with unknown labels are excluded by the shared training loader; candidate misses stay in Top-1. Do not filter no-positive or singleton groups for logistic stacking. Keep all student-test candidates for inference and attach reference labels only when scoring.

Base and combiner prediction times are reported separately. Their sum is available only when all four base timing records are present; missing timing is not treated as zero. The bundle stores the six meta features and the 32 base features separately. Register the completed stacker only after all artifacts and metrics have been saved; this never changes `active_model`.

In [3]:
def base_prediction_seconds(name, record, fold=None):
    path = record.get("metrics_path") if fold is None else record.get("cv_metrics_path")
    if fold is not None and not path and record.get("oof_path"):
        path = str(Path(record["oof_path"]).parent / "cv_metrics.csv")
    if not path or not Path(path).is_file():
        return None
    if fold is None:
        values = json.loads(Path(path).read_text())
    else:
        table = pd.read_csv(path)
        if "fold" not in table or not table["fold"].eq(fold).any():
            return None
        values = table[table["fold"] == fold].iloc[0].to_dict()
    for key in ["final_predict_s", "prediction_s", "predict_s", "test_predict_s"]:
        value = values.get(key)
        if value is not None and np.isfinite(float(value)):
            return float(value)
    return None


def run_stacking(registry):
    missing = missing_base_artifacts(registry)
    if missing:
        raise ValueError("Stacking is not ready: " + "; ".join(missing))
    base_records = {name: registry["models"][name] for name in base_model_order}
    source, points = load_shared_training()
    source = source[["trip_id", "group_id", "point_id", "segment_id", "label", "fold", "distance", "candidate_count"]]
    base_scores = {name: pd.read_csv(record["oof_path"], usecols=["group_id", "segment_id", "fold", "score"])
                   for name, record in base_records.items()}
    meta = build_meta_table(source, base_scores)
    del source, base_scores
    assert meta["label"].notna().all() and set(meta["label"].unique()) == {0, 1}
    assert meta.groupby("trip_id")["fold"].nunique().eq(1).all()
    started = datetime.now(ZoneInfo("Asia/Kolkata"))
    run_id = started.strftime("%Y%m%d_%H%M%S")
    output = Path("results/stacking") / run_id
    output.mkdir(parents=True, exist_ok=True)
    oof = np.full(len(meta), np.nan, dtype=float)
    records = []
    for fold in range(5):
        train = meta[meta["fold"] != fold]
        valid = meta[meta["fold"] == fold]
        assert set(train["trip_id"]).isdisjoint(valid["trip_id"])
        model = make_combiner()
        start = perf_counter()
        model.fit(train[meta_feature_columns], train["label"])
        fit_s = perf_counter() - start
        start = perf_counter()
        probability = point_softmax(model.decision_function(valid[meta_feature_columns]), valid["group_id"])
        predict_s = perf_counter() - start
        oof[valid.index] = probability
        base_times = {name: base_prediction_seconds(name, record, fold) for name, record in base_records.items()}
        total = predict_s + sum(base_times.values()) if all(v is not None for v in base_times.values()) else None
        metrics = score_points(valid, probability, points[points["fold"] == fold])
        records.append({"fold": fold, **metrics, "fit_s": fit_s, "combiner_predict_s": predict_s,
                        "base_predict_s": json.dumps(base_times), "total_predict_s": total, "predict_s": total})
        print("Meta fold", fold, "Top-1", metrics["top1"], flush=True)
    assert np.isfinite(oof).all()
    predictions = meta[["group_id", "segment_id", "fold"]].copy()
    predictions["score"] = oof
    predictions.to_csv(output / "oof_stacking.csv", index=False)
    cv = pd.DataFrame(records)
    cv.to_csv(output / "cv_metrics.csv", index=False)
    pooled = score_points(meta, oof, points)
    (output / "pooled_metrics.json").write_text(json.dumps(pooled, indent=2) + "\n")

    model = make_combiner()
    start = perf_counter()
    model.fit(meta[meta_feature_columns], meta["label"])
    final_fit_s = perf_counter() - start
    bundle = {"model": model, "meta_feature_columns": meta_feature_columns,
              "base_model_order": base_model_order, "model_type": "stacking",
              "base_feature_columns": shared_feature_columns, "base_models": base_records,
              "seed": seed, "excluded_trip_ids": training_exclusions,
              "fold_file": contract["fold_file"], "score_transform": "decision_function_then_point_softmax"}
    joblib.dump(bundle, output / "model.joblib")
    (output / "feature_columns.json").write_text(json.dumps(meta_feature_columns, indent=2) + "\n")

    test_source, test_points = load_shared_student_test()
    test_scores = {name: pd.read_csv(record["student_test_scores_path"], usecols=["group_id", "segment_id", "score"])
                   for name, record in base_records.items()}
    test_meta = build_meta_table(test_source, test_scores, require_fold=False)
    assert test_meta["label"].isna().all()
    start = perf_counter()
    test_probability = point_softmax(model.decision_function(test_meta[meta_feature_columns]), test_meta["group_id"])
    combiner_predict_s = perf_counter() - start
    test_predictions = test_meta[["group_id", "segment_id"]].copy()
    test_predictions["score"] = test_probability
    test_predictions.to_csv(output / "student_test_predictions.csv", index=False)
    test_metrics = score_student_test(test_meta, test_probability, test_points)
    base_times = {name: base_prediction_seconds(name, record) for name, record in base_records.items()}
    total = combiner_predict_s + sum(base_times.values()) if all(v is not None for v in base_times.values()) else None
    test_metrics.update({"final_fit_s": final_fit_s, "combiner_predict_s": combiner_predict_s,
                        "base_predict_s": base_times, "prediction_s": total,
                        "timing_scope": "four saved base inference times plus measured combiner and softmax; excludes feature generation"})
    (output / "student_test_metrics.json").write_text(json.dumps(test_metrics, indent=2) + "\n")
    (output / "run_config.json").write_text(json.dumps({
        "base_models": base_records, "base_model_order": base_model_order,
        "meta_feature_columns": meta_feature_columns, "seed": seed, "excluded_trip_ids": training_exclusions,
        "fold_scheme": contract["fold_scheme"],
        "cv_caveat": "Ordinary meta CV on global OOF; not fully nested base-pipeline assessment.",
        "student_test_tuning": False, "training_rows": len(meta)}, indent=2) + "\n")
    metric_names = ["top1", "macro_f1", "precision", "recall", "log_loss", "candidate_coverage"]
    row = {"run_id": run_id, "timestamp": started.isoformat(), "model": "stacking",
           "params": json.dumps({"combiner": "StandardScaler + LogisticRegression", "max_iter": 1000}),
           "seed": seed, "fold_scheme": contract["fold_scheme"] + "; ordinary meta CV on global OOF, not nested",
           "radius_m": contract["candidate_settings"]["radius_m"], "K": contract["candidate_settings"]["K"],
           "fit_s_mean": cv["fit_s"].mean(), "combiner_predict_s_mean": cv["combiner_predict_s"].mean(),
           "predict_s_mean": cv["total_predict_s"].mean() if cv["total_predict_s"].notna().all() else None,
           "final_fit_s": final_fit_s, "final_predict_s": total, "test_status": "scored_after_final_training",
           "output_dir": str(output)}
    for metric in metric_names:
        row[f"cv_{metric}_mean"] = cv[metric].mean()
        row[f"cv_{metric}_std"] = cv[metric].std()
        row[f"pooled_{metric}"] = pooled[metric]
        row[f"test_{metric}"] = test_metrics[metric]
    experiments = Path("results/experiments.csv")
    old = pd.read_csv(experiments) if experiments.exists() else pd.DataFrame()
    pd.concat([old, pd.DataFrame([row])], ignore_index=True).to_csv(experiments, index=False)
    record = {"status": "trained", "kind": "stacking", "model_path": str(output / "model.joblib"),
              "feature_columns_path": str(output / "feature_columns.json"), "base_model_order": base_model_order,
              "oof_path": str(output / "oof_stacking.csv"),
              "student_test_scores_path": str(output / "student_test_predictions.csv"),
              "metrics_path": str(output / "student_test_metrics.json"), "hmm_config_path": None}
    (output / "registry_record.json").write_text(json.dumps(record, indent=2) + "\n")
    register_model("stacking", record)
    print("Saved complete stacker. Active model unchanged; separate HMM tuning is still required.")
    display(cv[metric_names].agg(["mean", "std"]).T)
    print("Pooled OOF:", pooled, "| Student-test:", test_metrics)
    return record

## Readiness gate

When inputs are still missing, this cell only reports them. It creates no model, predictions, scores or registry entry. Run again after all four complete model/OOF handoffs have been registered.

`run_when_ready` is deliberately false while this integration is being prepared. Set it true only when the four complete handoffs are registered and you want to launch the real fit.

In [4]:
missing = missing_base_artifacts(registry)
if missing:
    print("Stacker not fitted. Required incoming artifacts:")
    for item in missing:
        print(" -", item)
elif run_when_ready:
    stacking_record = run_stacking(registry)
else:
    print("All base artifacts are available. Set run_when_ready = True to fit and score the real stacker.")

Stacker not fitted. Required incoming artifacts:
 - logistic_regression: status is awaiting_user_files
 - logistic_regression: missing model_path
 - logistic_regression: missing feature_columns_path
 - logistic_regression: missing oof_path
 - logistic_regression: missing student_test_scores_path
 - knn: status is awaiting_final_user_files_existing_notebook_is_pilot
 - knn: missing model_path
 - knn: missing feature_columns_path
 - knn: missing oof_path
 - knn: missing student_test_scores_path
 - decision_tree: status is integrating
 - decision_tree: missing model_path
 - decision_tree: missing oof_path
 - decision_tree: missing student_test_scores_path
